In [ ]:
import sys
sys.path.insert(0, "/home/kkingstoun/git/containers_admin2/compute-lib/mmpp")
print("mmpp path added:", sys.path[0])


# Vortex Dynamics Analysis - full numerical workflow

This notebook runs a full vortex post-processing workflow on a real simulation:

`/mnt/storage_6/project_data/pl0095-01/mateuszz/microlab/projects/marie_cuire_vortex_stt/workspace/scratch/minimalmodel/v1/gptpro_fast.zarr`

Pipeline covered:
1. Data loading and metadata inspection
2. Core tracking (maximum / centroid / gaussian)
3. Topology detection
4. Trajectory analysis (orbit, phase, directional spectrum)
5. Spectrum + mode classification
6. Events, signals, and energy post-processing
7. Nonlinear metrics + numerical to analytical bridge
8. Thiele adapters with auto-inferred parameters


## 1) Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr

from IPython.display import display

import mmpp

print("mmpp version:", mmpp.__version__)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 4)


def safe_tight_layout(fig=None):
    try:
        (fig or plt.gcf()).tight_layout()
    except RuntimeError:
        pass


## 2) Load simulation


In [ ]:
ZARR_PATH = "/mnt/storage_6/project_data/pl0095-01/mateuszz/microlab/projects/marie_cuire_vortex_stt/workspace/scratch/minimalmodel/v1/gptpro_fast.zarr"

scan = mmpp.MMPP(ZARR_PATH)
if len(scan) == 0:
    raise RuntimeError(f"No jobs found in: {ZARR_PATH}")

sim = scan[0]
z_root = zarr.open(ZARR_PATH, mode="r")

display(scan)
display(sim)

print("Top-level keys:", list(z_root.keys()))
if "m" in z_root:
    print("m shape:", z_root["m"].shape, "dtype:", z_root["m"].dtype)
if "table" in z_root:
    print("table columns:", len(list(z_root["table"].keys())))


In [ ]:
attrs = dict(sim.attrs)
attr_preview = pd.DataFrame([
    {"key": k, "value": str(v)} for k, v in sorted(attrs.items())
])
display(attr_preview.head(40))

if "table" in z_root:
    table = z_root["table"]
    table_info = []
    for key in table.keys():
        arr = table[key]
        table_info.append({"column": key, "shape": tuple(arr.shape), "dtype": str(arr.dtype)})
    display(pd.DataFrame(table_info).sort_values("column").reset_index(drop=True))


def find_column(table_group, aliases):
    lower_map = {str(k).lower(): str(k) for k in table_group.keys()}
    for alias in aliases:
        key = lower_map.get(str(alias).lower())
        if key is not None:
            return key
    return None


## 3) Build vortex interface


In [ ]:
data = sim.m
vortex = data.vortex

display(vortex)

print("dataset shape:", data.shape)
print("dataset dt [s]:", getattr(data, "dt", "n/a"))
print("dx, dy [m]:", attrs.get("dx"), attrs.get("dy"))


## 4) Core tracking (maximum / centroid / gaussian)


In [ ]:
tracking = {}
errors = {}
for method in ("maximum", "centroid", "gaussian"):
    try:
        tr = vortex.track(method=method, z_layer=0, force=True)
        tracking[method] = tr
    except Exception as exc:
        errors[method] = repr(exc)

if not tracking:
    raise RuntimeError(f"Tracking failed for all methods: {errors}")

traj = tracking.get("gaussian") or next(iter(tracking.values()))
print("selected trajectory method:", traj.method)

rows = []
for method, tr in tracking.items():
    mean_radius_nm = float(np.mean(tr.r) * 1e9) if tr.time.size else float("nan")
    mean_freq_ghz = float(np.mean(tr.instantaneous_frequency) / (2.0 * np.pi * 1e9)) if tr.time.size > 1 else float("nan")
    rows.append(
        {
            "method": method,
            "n_samples": int(tr.time.size),
            "rotation": tr.rotation_sense,
            "mean_radius_nm": mean_radius_nm,
            "mean_freq_ghz": mean_freq_ghz,
            "confidence_mean": float(np.mean(tr.confidence)) if tr.confidence.size else float("nan"),
        }
    )

display(pd.DataFrame(rows).sort_values("method").reset_index(drop=True))

if errors:
    print("tracking errors:", errors)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
traj.plt.xy(ax=axes[0])
axes[0].set_title("Tracked core: x(t), y(t)")
traj.plt.orbit_2d(ax=axes[1])
axes[1].set_title("Tracked orbit")
safe_tight_layout()
plt.show()


## 5) Optional check against table core positions (`ext_coreposx/y`)


In [ ]:
if "table" in z_root:
    table = z_root["table"]
    key_x = find_column(table, ("ext_coreposx", "VortexX", "vortexx", "coreposx"))
    key_y = find_column(table, ("ext_coreposy", "VortexY", "vortexy", "coreposy"))

    if key_x is not None and key_y is not None:
        x_ref = np.asarray(table[key_x][:], dtype=float)
        y_ref = np.asarray(table[key_y][:], dtype=float)

        n = int(min(traj.time.size, x_ref.size, y_ref.size))
        x_num = np.asarray(traj.x[:n], dtype=float)
        y_num = np.asarray(traj.y[:n], dtype=float)
        x_ref = x_ref[:n]
        y_ref = y_ref[:n]

        rmse_nm = float(np.sqrt(np.mean((x_num - x_ref) ** 2 + (y_num - y_ref) ** 2)) * 1e9)
        print(f"reference columns: {key_x}, {key_y}")
        print(f"alignment length : {n}")
        print(f"RMSE(num vs table): {rmse_nm:.4f} nm")

        fig, ax = plt.subplots(figsize=(5.6, 5.0))
        ax.plot(x_num * 1e9, y_num * 1e9, label="numerical track", lw=1.0)
        ax.plot(x_ref * 1e9, y_ref * 1e9, label="table core pos", lw=1.0, ls="--")
        ax.set_xlabel("x [nm]")
        ax.set_ylabel("y [nm]")
        ax.set_title("Orbit overlay: tracked vs table")
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.25)
        ax.legend()
        safe_tight_layout()
        plt.show()
    else:
        print("No ext_coreposx/y (or VortexX/Y) columns in table.")


## 6) Topology snapshots


In [ ]:
frame_ids = sorted(set([0, int(traj.time.size // 2), int(max(traj.time.size - 1, 0))]))

topo_rows = []
topo_by_frame = {}
for fid in frame_ids:
    topo = vortex.topology.detect(t=int(fid), method="finite_diff", z_layer=0, force=True)
    topo_by_frame[int(fid)] = topo
    topo_rows.append(
        {
            "frame": int(fid),
            "state": topo.state,
            "polarity": int(topo.polarity),
            "vorticity": int(topo.vorticity),
            "chirality": int(topo.chirality),
            "Q": float(topo.Q),
            "confidence": float(topo.confidence),
            "is_consistent": bool(topo.is_consistent),
        }
    )

display(pd.DataFrame(topo_rows))

mid = topo_by_frame[frame_ids[len(frame_ids) // 2]]
q_map = getattr(mid, "topological_density", None)
if q_map is not None and np.size(q_map):
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(np.asarray(q_map, dtype=float), origin="lower", cmap="coolwarm")
    ax.set_title("Topological density (middle frame)")
    plt.colorbar(im, ax=ax, label="q")
    safe_tight_layout()
    plt.show()


## 7) Trajectory analysis (orbit, phase, directional spectrum)


In [ ]:
orbit_fit = traj.analysis.orbit.fit(model="ellipse")
phase_wrapped = traj.analysis.phase.instantaneous(method="complex")
phase_unwrapped = np.unwrap(phase_wrapped)
freq_ghz = traj.analysis.phase.frequency(method="complex", unit="ghz")
directional = traj.analysis.spectrum.directional(method="welch")

print(f"orbit center [nm]: ({orbit_fit.center[0] * 1e9:.4f}, {orbit_fit.center[1] * 1e9:.4f})")
print(f"orbit radius [nm]: {orbit_fit.radius * 1e9:.4f}")
print(f"orbit eccentricity: {orbit_fit.eccentricity:.4f}")
print(f"phase-based median f [GHz]: {np.nanmedian(freq_ghz):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(traj.time * 1e9, phase_unwrapped / (2.0 * np.pi), lw=1.0)
axes[0].set_xlabel("t [ns]")
axes[0].set_ylabel("phase / 2pi [cycles]")
axes[0].set_title("Unwrapped phase")
axes[0].grid(True, alpha=0.25)

axes[1].plot(traj.time * 1e9, freq_ghz, lw=1.0)
axes[1].set_xlabel("t [ns]")
axes[1].set_ylabel("f_inst [GHz]")
axes[1].set_title("Instantaneous frequency")
axes[1].grid(True, alpha=0.25)

directional.plt.power_spectrum(ax=axes[2], unit="ghz")
axes[2].set_title("Directional spectrum (CCW/CW)")

safe_tight_layout()
plt.show()


## 8) Spectrum namespace + mode classification


In [ ]:
gyr = vortex.spectrum.gyration(method="welch")
breath = vortex.spectrum.breathing(method="welch")
sgram = vortex.spectrum.spectrogram(component="radius")

print(f"gyration peak [GHz]: {gyr.peak_frequency_ghz:.4f}")
print(f"breathing peak [GHz]: {breath.peak_frequency_ghz:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
gyr.plt.power_spectrum(ax=axes[0], as_ghz=True, log_scale=True)
axes[0].set_title("Gyration PSD")

breath.plt.power_spectrum(ax=axes[1], as_ghz=True, log_scale=True)
axes[1].set_title("Breathing PSD")

sgram.plt.spectrogram(ax=axes[2], as_ghz=True, db_scale=True)
axes[2].set_title("Radius spectrogram")

safe_tight_layout()
plt.show()

modes = vortex.modes.classify_all(max_modes=8, min_prominence=0.03)
mode_table = vortex.modes.plt.mode_table()
print("detected modes:", len(modes))
if mode_table:
    display(pd.DataFrame(mode_table))

fig, ax = plt.subplots(figsize=(8.0, 4.0))
vortex.modes.plt.mode_map(ax=ax)
safe_tight_layout()
plt.show()


## 9) Event detection


In [ ]:
pol_switch = vortex.events.polarity_switches(trajectory=traj)
state_switch = vortex.events.state_switches(trajectory=traj)
expulsion = vortex.events.core_expulsions(trajectory=traj)
dwell_g = vortex.events.dwell_times(trajectory=traj, state="G-state")

print("polarity switches:", len(pol_switch))
print("state switches   :", len(state_switch))
print("core expulsions  :", len(expulsion))
print("G-state dwells   :", dwell_g.count)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
vortex.events.plt.event_timeline(trajectory=traj, ax=axes[0])
axes[0].set_title("Event timeline")

if dwell_g.count > 0:
    dwell_g.plt.dwell_histogram(ax=axes[1])
else:
    axes[1].text(0.5, 0.5, "No dwell intervals detected", ha="center", va="center")
    axes[1].set_axis_off()

safe_tight_layout()
plt.show()


## 10) Signals and energy post-processing


In [ ]:
mr = vortex.signals.magnetoresistance(trajectory=traj)
voltage = vortex.signals.voltage(trajectory=traj, magnetoresistance=mr)
signal_psd = vortex.signals.power_spectrum(signal="voltage", trajectory=traj)

energy_ts = vortex.energy.time_resolved(force=True)
potential = vortex.energy.potential(trajectory=traj, method="auto", force=True)
pinning = vortex.energy.pinning(potential=potential, force=True)

print("signals method:", mr.method)
print("MR mean [Ohm]:", mr.mean_resistance_ohm)
print("V_rms [V]:", voltage.rms_voltage_v)
print("signal peak [GHz]:", signal_psd.peak_frequency_ghz)
print("energy channels:", energy_ts.available_channels)
print("pinning sites:", len(pinning.sites), "(method:", potential.method + ")")

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
mr.plt.time_trace(ax=axes[0, 0])
axes[0, 0].set_title("Magnetoresistance")

voltage.plt.time_trace(ax=axes[0, 1])
axes[0, 1].set_title("Voltage")

signal_psd.plt.power_spectrum(ax=axes[0, 2])
axes[0, 2].set_title("Voltage PSD")

if energy_ts.available_channels:
    energy_ts.plt.time_resolved(ax=axes[1, 0])
else:
    axes[1, 0].text(0.5, 0.5, "No energy channels found", ha="center", va="center")
    axes[1, 0].set_axis_off()

potential.plt.potential(ax=axes[1, 1], as_nev=True)
axes[1, 1].set_title("Effective potential")

pinning.plt.potential_with_sites(ax=axes[1, 2], as_nev=True)
axes[1, 2].set_title("Pinning minima")

safe_tight_layout()
plt.show()


## 11) Nonlinear analysis + bridge fit + Thiele adapters + auto f(J) calibration

In [ ]:
st = vortex.nonlinear.slavin_tiberkevich(trajectory=traj)
force_balance = vortex.nonlinear.force_balance(trajectory=traj)
bridge_fit = vortex.bridge.fit.thiele_from_trajectory(traj, damping=0.01)
cmp_proxy = traj.compare.with_(bridge_fit.simulated_trajectory, label=("numerical", "thiele_proxy"))

print(f"ST f0 [GHz]: {st.f_0_ghz:.4f}")
print(f"ST N [rad/s]: {st.N:.4e}")
print(f"ST linewidth [MHz]: {st.linewidth_hz * 1e-6:.4f}")
print("linewidth_resolution_limited:", st.linewidth_resolution_limited)
print(f"bridge delta_f_mean [Hz]: {cmp_proxy.metrics.delta_f_mean:.4e}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
cmp_proxy.plot.overlay_orbit(ax=axes[0])
axes[0].set_title("Numerical vs Thiele proxy (bridge fit)")

force_balance.plt.force_balance(ax=axes[1], as_norm=True)
axes[1].set_title("Thiele force balance")

safe_tight_layout()
plt.show()

cpp_adapter = vortex.model.thiele.cpp()
cip_adapter = vortex.model.thiele.cip()

model_rows = [
    {"parameter": "Ms [A/m]", "value": cpp_adapter.model.material.Ms, "source": "attrs: Ms/Msat"},
    {"parameter": "alpha", "value": cpp_adapter.model.material.alpha, "source": "attrs: alpha"},
    {"parameter": "P", "value": cpp_adapter.model.material.P, "source": "attrs: P/Pol"},
    {"parameter": "A [J/m]", "value": cpp_adapter.model.material.A, "source": "attrs: Aex/A"},
    {"parameter": "R [m]", "value": cpp_adapter.model.geom.R, "source": "dataset shape + dx/dy"},
    {"parameter": "L [m]", "value": cpp_adapter.model.geom.L, "source": "attrs: thickness/L/dz*Nz"},
    {"parameter": "polarity", "value": cpp_adapter.model.polarity, "source": "attrs: polarity/p"},
    {"parameter": "omega0 [rad/s]", "value": cpp_adapter.model.omega0, "source": "omega0_novosad(material, geom)"},
]
display(pd.DataFrame(model_rows))

if traj.time.size > 1:
    dt = float(np.median(np.diff(traj.time)))
else:
    dt = 1e-12
k = max(int(traj.time.size) - 1, 1)
t_end = float(np.nextafter(np.float64(dt) * np.float64(k), np.inf))

traj_cpp = None
traj_cip = None
cmp_cpp = None
cmp_cip = None

try:
    traj_cpp = cpp_adapter.simulate(t_span=(0.0, t_end), dt=dt, J_func="auto_from_table")
    traj_cip = cip_adapter.simulate(t_span=(0.0, t_end), dt=dt, J_func="auto_from_table")

    cmp_cpp = traj.compare.with_(traj_cpp, label=("numerical", "thiele_cpp"))
    cmp_cip = traj.compare.with_(traj_cip, label=("numerical", "thiele_cip"))

    print(f"delta_f_mean num-vs-cpp [Hz]: {cmp_cpp.metrics.delta_f_mean:.4e}")
    print(f"delta_f_mean num-vs-cip [Hz]: {cmp_cip.metrics.delta_f_mean:.4e}")

    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    cmp_cpp.plot.overlay_orbit(ax=ax)
    ax.plot(traj_cip.x, traj_cip.y, ls=":", lw=1.1, label="thiele_cip")
    ax.legend()
    ax.set_title("Numerical vs analytical adapters")
    safe_tight_layout()
    plt.show()
except Exception as exc:
    print("Analytical adapter simulation failed:", repr(exc))

# ---------------------------------------------------------------------
# Automatic CPP Thiele f(J) calibration from micromagnetic data
# ---------------------------------------------------------------------

J_ATTR_ALIASES = ("J", "j", "Jdc", "J_dc", "current_density", "currentdensity")
J_TABLE_ALIASES = ("J", "j", "Jdc", "J_dc", "current_density", "CurrentDensity")
MAX_SCAN_JOBS_FOR_FJ = 32


def _read_float_from_mapping(mapping, aliases):
    if mapping is None:
        return None
    try:
        lower = {str(k).lower(): k for k in mapping.keys()}
    except Exception:
        return None

    for alias in aliases:
        key = lower.get(str(alias).lower())
        if key is None:
            continue
        try:
            value = float(mapping[key])
        except (TypeError, ValueError):
            continue
        if np.isfinite(value):
            return value
    return None


def _read_series_from_table(table_group, aliases):
    if table_group is None:
        return None
    key = find_column(table_group, aliases)
    if key is None:
        return None
    try:
        values = np.asarray(table_group[key][:], dtype=float).reshape(-1)
    except Exception:
        return None
    if values.size == 0:
        return None
    return key, values


def _extract_job_current_density(job):
    attrs_local = getattr(job, "attrs", {})
    from_attrs = _read_float_from_mapping(attrs_local, J_ATTR_ALIASES)
    if from_attrs is not None:
        return float(from_attrs), "attrs"

    table_group = None
    try:
        table_group = job["table"]
    except Exception:
        table_group = None

    payload = _read_series_from_table(table_group, J_TABLE_ALIASES)
    if payload is not None:
        key, series = payload
        finite = np.isfinite(series)
        if np.any(finite):
            return float(np.nanmedian(series[finite])), f"table:{key}"

    return float("nan"), "missing"


def _extract_job_frequency_hz(job, tracking_method="gaussian"):
    data_local = getattr(job, "m", None)
    if data_local is None:
        return float("nan"), "missing_dataset"

    vortex_local = data_local.vortex
    try:
        traj_local = vortex_local.track(method=tracking_method, z_layer=0, force=True)
    except Exception:
        try:
            traj_local = vortex_local.track(method="maximum", z_layer=0, force=True)
        except Exception as exc:
            return float("nan"), f"track_failed:{exc!r}"

    omega_inst = np.asarray(
        getattr(traj_local, "instantaneous_frequency", np.array([])),
        dtype=float,
    ).reshape(-1)
    finite = np.isfinite(omega_inst)
    if np.any(finite):
        f_hz = float(np.nanmedian(np.abs(omega_inst[finite])) / (2.0 * np.pi))
        return f_hz, f"track:{traj_local.method}"

    try:
        gyr_local = vortex_local.spectrum.gyration(method="welch")
        return float(gyr_local.peak_frequency_ghz * 1e9), "welch_peak"
    except Exception as exc:
        return float("nan"), f"spectrum_failed:{exc!r}"


def _build_fj_points_from_scan(scan_obj, max_jobs=32):
    cols = ["job_idx", "J_A_per_m2", "f_hz", "J_source", "f_source"]
    n_jobs_total = int(len(scan_obj))
    if n_jobs_total == 0:
        return pd.DataFrame(columns=cols)

    if n_jobs_total > int(max_jobs):
        indices = np.linspace(0, n_jobs_total - 1, int(max_jobs), dtype=int)
        indices = np.unique(indices)
    else:
        indices = np.arange(n_jobs_total, dtype=int)

    rows = []
    for idx in indices:
        job = scan_obj[int(idx)]
        j_val, j_source = _extract_job_current_density(job)
        f_val, f_source = _extract_job_frequency_hz(job)
        rows.append(
            {
                "job_idx": int(idx),
                "J_A_per_m2": float(j_val),
                "f_hz": float(f_val),
                "J_source": j_source,
                "f_source": f_source,
            }
        )
    return pd.DataFrame(rows, columns=cols)


def _build_fj_points_from_single_run(traj_obj, table_group, n_bins=12, min_samples=8):
    cols = ["job_idx", "J_A_per_m2", "f_hz", "J_source", "f_source"]
    payload = _read_series_from_table(table_group, J_TABLE_ALIASES)
    if payload is None:
        return pd.DataFrame(columns=cols)

    j_key, j_series = payload
    omega_inst = np.asarray(
        getattr(traj_obj, "instantaneous_frequency", np.array([])),
        dtype=float,
    ).reshape(-1)
    if omega_inst.size == 0:
        return pd.DataFrame(columns=cols)

    n = int(min(j_series.size, omega_inst.size))
    if n < int(min_samples) * 3:
        return pd.DataFrame(columns=cols)

    j = np.asarray(j_series[:n], dtype=float)
    f = np.abs(np.asarray(omega_inst[:n], dtype=float)) / (2.0 * np.pi)

    mask = np.isfinite(j) & np.isfinite(f)
    if np.count_nonzero(mask) < int(min_samples) * 3:
        return pd.DataFrame(columns=cols)

    j = j[mask]
    f = f[mask]
    j_span = float(np.nanmax(j) - np.nanmin(j))
    if not np.isfinite(j_span) or j_span <= 0.0:
        return pd.DataFrame(columns=cols)

    quantiles = np.linspace(0.0, 1.0, int(max(n_bins, 4)) + 1)
    edges = np.unique(np.quantile(j, quantiles))
    if edges.size < 4:
        return pd.DataFrame(columns=cols)

    bin_ids = np.digitize(j, edges[1:-1], right=False)
    rows = []
    for bin_idx in range(edges.size - 1):
        sel = bin_ids == bin_idx
        if np.count_nonzero(sel) < int(min_samples):
            continue
        rows.append(
            {
                "job_idx": -1,
                "J_A_per_m2": float(np.nanmedian(j[sel])),
                "f_hz": float(np.nanmedian(f[sel])),
                "J_source": f"single_run_binned:{j_key}",
                "f_source": "track:instantaneous_frequency",
            }
        )

    return pd.DataFrame(rows, columns=cols)


def _prepare_fj_for_fit(df):
    if df is None or df.empty:
        return None

    out = df.copy()
    out = out[np.isfinite(out["J_A_per_m2"].to_numpy(dtype=float))]
    out = out[np.isfinite(out["f_hz"].to_numpy(dtype=float))]
    out = out.reset_index(drop=True)
    if out.empty:
        return None

    j_values = out["J_A_per_m2"].to_numpy(dtype=float)
    scale = max(float(np.nanmax(np.abs(j_values))), 1.0)
    unique_norm_j = np.unique(np.round(j_values / scale, 6))
    if unique_norm_j.size < 3:
        return None

    return out.sort_values("J_A_per_m2").reset_index(drop=True)


fj_points_scan = _build_fj_points_from_scan(scan, max_jobs=MAX_SCAN_JOBS_FOR_FJ)
if "table" in z_root:
    fj_points_single = _build_fj_points_from_single_run(traj, z_root["table"])
else:
    fj_points_single = pd.DataFrame(columns=["job_idx", "J_A_per_m2", "f_hz", "J_source", "f_source"])

fj_points_fit = _prepare_fj_for_fit(fj_points_scan)
fj_fit_source = "scan_jobs"
if fj_points_fit is None:
    fj_points_fit = _prepare_fj_for_fit(fj_points_single)
    fj_fit_source = "single_run_bins"

print("\nThiele f(J) auto-fit data diagnostics:")
print(f"scan points         : {len(fj_points_scan)}")
print(f"single-run bin points: {len(fj_points_single)}")
print(
    "fit source          :",
    fj_fit_source if fj_points_fit is not None else "insufficient_data",
)

if fj_points_fit is not None:
    display(fj_points_fit)
else:
    print("No valid set of >=3 distinct (J, f) points found.")

thiele_fj_fit = None
cmp_cpp_fitted = None

if fj_points_fit is not None:
    J_data = fj_points_fit["J_A_per_m2"].to_numpy(dtype=float)
    f_data_hz = fj_points_fit["f_hz"].to_numpy(dtype=float)

    try:
        thiele_fj_fit = mmpp.analytical.fit_omega0_N_to_fJ(
            J_data,
            f_data_hz,
            material=cpp_adapter.model.material,
            geom=cpp_adapter.model.geom,
            polarity=int(cpp_adapter.model.polarity),
            initial_omega0=float(cpp_adapter.model.omega0),
            initial_N=float(cpp_adapter.model.N),
            fit_domega0_dJ=True,
            fit_chi_scale=True,
            allow_edge=True,
        )

        fit_rows = [
            {"parameter": "omega0_fit [rad/s]", "value": thiele_fj_fit.omega0},
            {"parameter": "N_fit [-]", "value": thiele_fj_fit.N},
            {"parameter": "domega0_dJ_fit [rad/s/(A/m^2)]", "value": thiele_fj_fit.domega0_dJ},
            {"parameter": "chi_scale_fit [-]", "value": thiele_fj_fit.chi_scale},
            {"parameter": "rmse_fit [MHz]", "value": thiele_fj_fit.rmse_hz * 1e-6},
            {"parameter": "fit_status", "value": thiele_fj_fit.status},
        ]
        display(pd.DataFrame(fit_rows))

        fig, ax = plt.subplots(figsize=(6.8, 4.6))
        thiele_fj_fit.plt.frequency_vs_current(ax=ax)
        ax.set_title(f"CPP Thiele f(J) auto-fit ({fj_fit_source})")
        safe_tight_layout()
        plt.show()

        current_sim, current_sim_source = _extract_job_current_density(sim)
        if np.isfinite(current_sim):
            cpp_model_fit = mmpp.analytical.CPPThieleModel(
                material=cpp_adapter.model.material,
                geom=cpp_adapter.model.geom,
                omega0=float(thiele_fj_fit.omega0),
                N=float(thiele_fj_fit.N),
                polarity=int(cpp_adapter.model.polarity),
                domega0_dJ=float(thiele_fj_fit.domega0_dJ),
                chi_scale=float(thiele_fj_fit.chi_scale),
            )
            f_pred = cpp_model_fit.predict_frequency_dc(current_sim, allow_edge=True)
            if f_pred is not None and np.isfinite(f_pred):
                print(
                    f"predicted f at current ({current_sim_source}) [GHz]: {float(f_pred) * 1e-9:.4f}"
                )

        try:
            cpp_fitted_adapter = vortex.model.thiele.cpp(
                omega0=float(thiele_fj_fit.omega0),
                N=float(thiele_fj_fit.N),
                domega0_dJ=float(thiele_fj_fit.domega0_dJ),
                chi_scale=float(thiele_fj_fit.chi_scale),
            )
            traj_cpp_fitted = cpp_fitted_adapter.simulate(
                t_span=(0.0, t_end),
                dt=dt,
                J_func="auto_from_table",
            )
            cmp_cpp_fitted = traj.compare.with_(
                traj_cpp_fitted,
                label=("numerical", "thiele_cpp_fitted"),
            )
            print(
                f"delta_f_mean num-vs-cpp_fitted [Hz]: {cmp_cpp_fitted.metrics.delta_f_mean:.4e}"
            )

            fig, ax = plt.subplots(figsize=(6.0, 5.0))
            cmp_cpp_fitted.plot.overlay_orbit(ax=ax)
            ax.set_title("Numerical vs calibrated CPP Thiele")
            safe_tight_layout()
            plt.show()
        except Exception as exc:
            print("Calibrated CPP trajectory comparison failed:", repr(exc))

    except Exception as exc:
        print("Automatic f(J) fit failed:", repr(exc))


# ---------------------------------------------------------------------
# Current-dependence engineering dashboard (interactive-like, notebook mode)
# ---------------------------------------------------------------------

engineering_sweep_df = pd.DataFrame()
critical_currents_df = pd.DataFrame()
target_currents_df = pd.DataFrame()
interactive_params_df = pd.DataFrame()
cpp_internal_df = pd.DataFrame()

j_threshold_a_per_m2 = float("nan")
j_onset_scan_a_per_m2 = float("nan")
j_edge_a_per_m2 = float("nan")
j_operating_margin_a_per_m2 = float("nan")
thiele_engineering_model_source = "base_cpp_adapter"


def _j_to_ma_cm2(j_val):
    return np.asarray(j_val, dtype=float) / 1e10


def _j_to_mA(j_val, area_m2):
    return np.asarray(j_val, dtype=float) * float(area_m2) * 1e3


def _first_j_where(j_values, mask):
    idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    if idx.size == 0:
        return float("nan")
    return float(np.asarray(j_values, dtype=float)[idx[0]])


def _scalar_ma_cm2(j_val):
    if not np.isfinite(j_val):
        return float("nan")
    return float(j_val / 1e10)


def _scalar_mA(j_val, area_m2):
    if not np.isfinite(j_val):
        return float("nan")
    return float(j_val * float(area_m2) * 1e3)


cpp_model_engineering = None
if thiele_fj_fit is not None:
    try:
        cpp_model_engineering = mmpp.analytical.CPPThieleModel(
            material=cpp_adapter.model.material,
            geom=cpp_adapter.model.geom,
            omega0=float(thiele_fj_fit.omega0),
            N=float(thiele_fj_fit.N),
            polarity=int(cpp_adapter.model.polarity),
            domega0_dJ=float(thiele_fj_fit.domega0_dJ),
            chi_scale=float(thiele_fj_fit.chi_scale),
        )
        thiele_engineering_model_source = "fitted_fJ"
    except Exception as exc:
        print("Engineering dashboard fallback to base CPP model:", repr(exc))

if cpp_model_engineering is None:
    cpp_model_engineering = cpp_adapter.model

try:
    geo_eng = cpp_model_engineering.geom
    mat_eng = cpp_model_engineering.material
    area_eng_m2 = float(np.pi * geo_eng.R**2)

    current_sim_j, current_sim_j_source = _extract_job_current_density(sim)

    j_observed_chunks = []
    for df_local in (fj_points_fit, fj_points_scan, fj_points_single):
        if isinstance(df_local, pd.DataFrame) and (not df_local.empty) and ("J_A_per_m2" in df_local.columns):
            vals = np.asarray(df_local["J_A_per_m2"], dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size:
                j_observed_chunks.append(vals)
    if j_observed_chunks:
        j_observed = np.concatenate(j_observed_chunks)
    else:
        j_observed = np.array([], dtype=float)

    j_abs_ref = 0.0
    if j_observed.size:
        j_abs_ref = max(j_abs_ref, float(np.nanmax(np.abs(j_observed))))
    if np.isfinite(current_sim_j):
        j_abs_ref = max(j_abs_ref, abs(float(current_sim_j)))

    j_threshold = float(cpp_model_engineering.threshold_current_dc())
    if np.isfinite(j_threshold):
        j_threshold_a_per_m2 = float(j_threshold)
        j_abs_ref = max(j_abs_ref, abs(float(j_threshold)))

    if j_abs_ref <= 0.0:
        j_abs_ref = 1e10

    j_min_scan = 0.0
    if j_observed.size:
        observed_min = float(np.nanmin(j_observed))
        if np.isfinite(observed_min) and observed_min < 0.0:
            j_min_scan = 1.2 * observed_min

    j_max_by_threshold = 0.0
    if np.isfinite(j_threshold) and abs(j_threshold) > 0.0:
        j_max_by_threshold = 4.0 * abs(j_threshold)

    j_max_scan = max(1.2 * j_abs_ref, j_max_by_threshold, 1e10)
    if not np.isfinite(j_max_scan) or j_max_scan <= j_min_scan:
        j_min_scan = 0.0
        j_max_scan = max(1e10, 1.2 * j_abs_ref)

    n_scan = 500 if j_min_scan < 0.0 else 400
    j_grid = np.linspace(j_min_scan, j_max_scan, int(max(n_scan, 200)))
    j_grid_ma = _j_to_ma_cm2(j_grid)

    u_stop = 0.98
    u_clamped = np.full(j_grid.shape, np.nan, dtype=float)
    u_free = np.full(j_grid.shape, np.nan, dtype=float)
    f_pred_hz = np.full(j_grid.shape, np.nan, dtype=float)
    chi_rad_s = np.full(j_grid.shape, np.nan, dtype=float)
    omega0_eff_rad_s = np.full(j_grid.shape, np.nan, dtype=float)
    balance_rad_s = np.full(j_grid.shape, np.nan, dtype=float)

    for idx, j_val in enumerate(j_grid):
        try:
            chi_val = float(cpp_model_engineering.chi(float(j_val)))
        except Exception:
            chi_val = float("nan")
        try:
            w0_eff_val = float(cpp_model_engineering.omega0_eff(float(j_val)))
        except Exception:
            w0_eff_val = float("nan")

        u_free_val = cpp_model_engineering.steady_state_u(float(j_val), allow_edge=False, u_stop=u_stop)
        u_clip_val = cpp_model_engineering.steady_state_u(float(j_val), allow_edge=True, u_stop=u_stop)
        f_val = cpp_model_engineering.predict_frequency_dc(float(j_val), allow_edge=True)

        if u_free_val is not None and np.isfinite(u_free_val):
            u_free[idx] = float(u_free_val)
        if u_clip_val is not None and np.isfinite(u_clip_val):
            u_clamped[idx] = float(u_clip_val)
        if f_val is not None and np.isfinite(f_val):
            f_pred_hz[idx] = float(f_val)

        chi_rad_s[idx] = chi_val
        omega0_eff_rad_s[idx] = w0_eff_val

        if np.isfinite(u_clamped[idx]):
            balance_rad_s[idx] = float(
                cpp_model_engineering.d(float(u_clamped[idx]))
                * cpp_model_engineering.omega(float(u_clamped[idx]), float(j_val))
            )

    regime = np.full(j_grid.shape, "damped", dtype=object)
    mask_active = np.isfinite(u_clamped)
    mask_free = np.isfinite(u_free)
    mask_edge = mask_active & (~mask_free)
    regime[mask_free] = "auto_oscillation"
    regime[mask_edge] = "edge_clamped"

    engineering_sweep_df = pd.DataFrame(
        {
            "J_A_per_m2": j_grid,
            "J_MA_per_cm2": j_grid_ma,
            "I_mA": _j_to_mA(j_grid, area_eng_m2),
            "f_pred_hz": f_pred_hz,
            "f_pred_ghz": f_pred_hz * 1e-9,
            "u0_clamped": u_clamped,
            "u0_free": u_free,
            "chi_rad_s": chi_rad_s,
            "omega0_eff_rad_s": omega0_eff_rad_s,
            "balance_rad_s": balance_rad_s,
            "regime": regime,
        }
    )

    j_onset_scan_a_per_m2 = _first_j_where(j_grid, mask_active)
    j_edge_a_per_m2 = _first_j_where(j_grid, mask_edge)
    if np.isfinite(j_threshold_a_per_m2) and np.isfinite(j_edge_a_per_m2) and (j_edge_a_per_m2 > j_threshold_a_per_m2):
        j_operating_margin_a_per_m2 = float(j_edge_a_per_m2 - j_threshold_a_per_m2)

    critical_rows = [
        {
            "quantity": "J_threshold_dc",
            "J_A_per_m2": float(j_threshold_a_per_m2),
            "J_MA_per_cm2": _scalar_ma_cm2(j_threshold_a_per_m2),
            "I_mA": _scalar_mA(j_threshold_a_per_m2, area_eng_m2),
            "note": "Analytical threshold: chi(J)=d0*omega0",
        },
        {
            "quantity": "J_onset_from_scan",
            "J_A_per_m2": float(j_onset_scan_a_per_m2),
            "J_MA_per_cm2": _scalar_ma_cm2(j_onset_scan_a_per_m2),
            "I_mA": _scalar_mA(j_onset_scan_a_per_m2, area_eng_m2),
            "note": "First J with finite steady-state orbit",
        },
        {
            "quantity": f"J_edge_clamp_u{u_stop:.2f}",
            "J_A_per_m2": float(j_edge_a_per_m2),
            "J_MA_per_cm2": _scalar_ma_cm2(j_edge_a_per_m2),
            "I_mA": _scalar_mA(j_edge_a_per_m2, area_eng_m2),
            "note": "Start of edge-clamped regime",
        },
        {
            "quantity": "J_operating_margin",
            "J_A_per_m2": float(j_operating_margin_a_per_m2),
            "J_MA_per_cm2": _scalar_ma_cm2(j_operating_margin_a_per_m2),
            "I_mA": _scalar_mA(j_operating_margin_a_per_m2, area_eng_m2),
            "note": "Approx. margin: J_edge - J_th",
        },
        {
            "quantity": "J_current_in_dataset",
            "J_A_per_m2": float(current_sim_j),
            "J_MA_per_cm2": _scalar_ma_cm2(current_sim_j),
            "I_mA": _scalar_mA(current_sim_j, area_eng_m2),
            "note": f"Source: {current_sim_j_source}",
        },
    ]

    if isinstance(fj_points_fit, pd.DataFrame) and (not fj_points_fit.empty):
        j_fit_min = float(np.nanmin(np.asarray(fj_points_fit["J_A_per_m2"], dtype=float)))
        j_fit_max = float(np.nanmax(np.asarray(fj_points_fit["J_A_per_m2"], dtype=float)))
        critical_rows.extend(
            [
                {
                    "quantity": "J_fit_min",
                    "J_A_per_m2": j_fit_min,
                    "J_MA_per_cm2": _scalar_ma_cm2(j_fit_min),
                    "I_mA": _scalar_mA(j_fit_min, area_eng_m2),
                    "note": f"f(J) fit source: {fj_fit_source}",
                },
                {
                    "quantity": "J_fit_max",
                    "J_A_per_m2": j_fit_max,
                    "J_MA_per_cm2": _scalar_ma_cm2(j_fit_max),
                    "I_mA": _scalar_mA(j_fit_max, area_eng_m2),
                    "note": f"f(J) fit source: {fj_fit_source}",
                },
            ]
        )

    critical_currents_df = pd.DataFrame(critical_rows)

    # Target-current helper table (inverse problem: J for target f)
    target_freqs_hz = []
    if isinstance(fj_points_fit, pd.DataFrame) and (not fj_points_fit.empty):
        f_vals = np.asarray(fj_points_fit["f_hz"], dtype=float)
        f_vals = f_vals[np.isfinite(f_vals) & (f_vals > 0.0)]
        if f_vals.size >= 3:
            target_freqs_hz = list(np.quantile(f_vals, [0.2, 0.5, 0.8]))

    if not target_freqs_hz:
        f_valid = f_pred_hz[np.isfinite(f_pred_hz) & (f_pred_hz > 0.0)]
        if f_valid.size >= 3:
            target_freqs_hz = list(np.quantile(f_valid, [0.2, 0.5, 0.8]))

    target_rows = []
    if target_freqs_hz:
        t_unique = sorted({float(v) for v in target_freqs_hz if np.isfinite(v) and v > 0.0})
        j_low_opt = max(0.0, float(j_min_scan))
        j_high_opt = float(j_max_scan)
        if j_high_opt > j_low_opt:
            for f_target in t_unique:
                try:
                    opt = cpp_model_engineering.optimize_current_for_target_frequency(
                        float(f_target),
                        J_bounds=(j_low_opt, j_high_opt),
                        allow_edge=True,
                    )
                    target_rows.append(
                        {
                            "target_f_ghz": float(f_target * 1e-9),
                            "J_opt_A_per_m2": float(opt.current_density_a_per_m2),
                            "J_opt_MA_per_cm2": _scalar_ma_cm2(float(opt.current_density_a_per_m2)),
                            "I_opt_mA": _scalar_mA(float(opt.current_density_a_per_m2), area_eng_m2),
                            "f_pred_ghz": float(opt.predicted_frequency_hz * 1e-9),
                            "abs_error_mhz": float(opt.objective_value_hz * 1e-6),
                            "success": bool(opt.success),
                            "status": str(opt.status),
                        }
                    )
                except Exception as exc:
                    target_rows.append(
                        {
                            "target_f_ghz": float(f_target * 1e-9),
                            "J_opt_A_per_m2": float("nan"),
                            "J_opt_MA_per_cm2": float("nan"),
                            "I_opt_mA": float("nan"),
                            "f_pred_ghz": float("nan"),
                            "abs_error_mhz": float("nan"),
                            "success": False,
                            "status": f"failed:{exc!r}",
                        }
                    )

    target_currents_df = pd.DataFrame(target_rows)

    # Plot bundle similar to interactive fast mode + engineering extras
    fig, axes = plt.subplots(1, 3, figsize=(17.0, 4.9))

    axes[0].plot(j_grid_ma, f_pred_hz * 1e-9, lw=1.5, label=f"CPP model ({thiele_engineering_model_source})")
    if isinstance(fj_points_fit, pd.DataFrame) and (not fj_points_fit.empty):
        axes[0].scatter(
            fj_points_fit["J_A_per_m2"].to_numpy(dtype=float) * 1e-10,
            fj_points_fit["f_hz"].to_numpy(dtype=float) * 1e-9,
            s=18,
            alpha=0.8,
            label=f"micromagnetic data ({fj_fit_source})",
        )

    if np.isfinite(j_threshold_a_per_m2):
        axes[0].axvline(j_threshold_a_per_m2 * 1e-10, ls="--", lw=1.0, color="tab:red", label="J_th")
    if np.isfinite(j_edge_a_per_m2):
        axes[0].axvline(j_edge_a_per_m2 * 1e-10, ls=":", lw=1.0, color="tab:purple", label="J_edge")
    if np.isfinite(current_sim_j):
        axes[0].axvline(current_sim_j * 1e-10, ls="-.", lw=1.0, color="tab:gray", label="J_dataset")

    axes[0].set_xlabel("Current density J [MA/cm^2]")
    axes[0].set_ylabel("Frequency [GHz]")
    axes[0].set_title("Current-frequency map")
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(fontsize=8)

    axes[1].plot(j_grid_ma, u_clamped, lw=1.4, label="u0 (allow_edge=True)")
    axes[1].plot(j_grid_ma, u_free, lw=1.1, ls="--", label="u0 (strict)")
    axes[1].axhline(u_stop, ls=":", lw=1.0, color="tab:red", label=f"u_stop={u_stop:.2f}")
    axes[1].set_xlabel("Current density J [MA/cm^2]")
    axes[1].set_ylabel("Normalized radius u0 = r0/R")
    axes[1].set_title("Orbit amplitude and edge clamp")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(fontsize=8)

    axes[2].plot(j_grid_ma, chi_rad_s / (2.0 * np.pi * 1e9), lw=1.2, label="chi(J)/2pi")
    axes[2].plot(j_grid_ma, balance_rad_s / (2.0 * np.pi * 1e9), lw=1.2, label="d(u0)*omega(u0,J)/2pi")
    axes[2].set_xlabel("Current density J [MA/cm^2]")
    axes[2].set_ylabel("Rate [GHz-equivalent]")
    axes[2].set_title("Pump vs damping balance")
    axes[2].grid(True, alpha=0.25)
    axes[2].legend(fontsize=8)

    safe_tight_layout()
    plt.show()

    print("\nCritical currents and operating margins:")
    display(critical_currents_df)

    if not target_currents_df.empty:
        print("Target-frequency -> optimal current (inverse design):")
        display(target_currents_df)

    # Mirror of interactive-dashboard control parameters
    chirality_attr = _read_float_from_mapping(attrs, ("chirality", "c"))
    if chirality_attr is None:
        chirality_val = int(cpp_model_engineering.field_cal.chirality)
    else:
        chirality_val = int(np.sign(chirality_attr) or 1)

    oersted_mhz_per_10ma_cm2 = float(cpp_model_engineering.domega0_dJ * 1e11 / (2.0 * np.pi * 1e6))

    interactive_rows = [
        {"tab": "Geom&Mat", "control": "w_model_type", "value": "CPP", "unit": "-", "source": "notebook workflow"},
        {"tab": "Geom&Mat", "control": "w_geom_mode", "value": "disk", "unit": "-", "source": "dataset geometry"},
        {"tab": "Geom&Mat", "control": "w_disk_d", "value": float(2.0 * geo_eng.R * 1e9), "unit": "nm", "source": "inferred geometry"},
        {"tab": "Geom&Mat", "control": "w_size_x", "value": float("nan"), "unit": "nm", "source": "N/A for disk"},
        {"tab": "Geom&Mat", "control": "w_size_y", "value": float("nan"), "unit": "nm", "source": "N/A for disk"},
        {"tab": "Geom&Mat", "control": "w_thick", "value": float(geo_eng.L * 1e9), "unit": "nm", "source": "attrs thickness/L/dz*Nz"},
        {"tab": "Geom&Mat", "control": "w_ms", "value": float(mat_eng.Ms * 1e-3), "unit": "kA/m", "source": "attrs Ms/Msat"},
        {"tab": "Geom&Mat", "control": "w_alpha", "value": float(mat_eng.alpha), "unit": "-", "source": "attrs alpha"},

        {"tab": "STT", "control": "w_current", "value": _scalar_mA(current_sim_j, area_eng_m2), "unit": "mA", "source": f"{current_sim_j_source} -> J*area"},
        {"tab": "STT", "control": "J_current", "value": _scalar_ma_cm2(current_sim_j), "unit": "MA/cm^2", "source": current_sim_j_source},
        {"tab": "STT", "control": "w_pol", "value": float(mat_eng.P), "unit": "-", "source": "attrs P/Pol"},
        {"tab": "STT", "control": "w_lambda", "value": 1.0, "unit": "-", "source": "interactive default"},
        {"tab": "STT", "control": "w_pz", "value": 1.0, "unit": "-", "source": "interactive default"},
        {"tab": "STT", "control": "w_beta", "value": float(mat_eng.beta), "unit": "-", "source": "material beta (CIP)"},
        {"tab": "STT", "control": "w_cip_angle", "value": 0.0, "unit": "deg", "source": "interactive default"},
        {"tab": "STT", "control": "w_p", "value": int(cpp_model_engineering.polarity), "unit": "-", "source": "attrs polarity/p"},
        {"tab": "STT", "control": "w_c", "value": int(chirality_val), "unit": "-", "source": "attrs chirality/c or default"},
        {"tab": "STT", "control": "w_angle", "value": 0.0, "unit": "deg", "source": "interactive default"},

        {"tab": "Fields", "control": "w_bx", "value": float(cpp_model_engineering.field.Bx_T * 1e3), "unit": "mT", "source": "model field"},
        {"tab": "Fields", "control": "w_by", "value": float(cpp_model_engineering.field.By_T * 1e3), "unit": "mT", "source": "model field"},
        {"tab": "Fields", "control": "w_bz", "value": float(cpp_model_engineering.field.Bz_T * 1e3), "unit": "mT", "source": "model field"},
        {"tab": "Fields", "control": "w_oersted", "value": oersted_mhz_per_10ma_cm2, "unit": "MHz per 10 MA/cm^2", "source": "from domega0_dJ"},
        {"tab": "Fields", "control": "w_field_mode", "value": "DC", "unit": "-", "source": "notebook analytical sweep"},
        {"tab": "Fields", "control": "w_bac_freq", "value": 1.0, "unit": "GHz", "source": "interactive default"},
        {"tab": "Fields", "control": "w_bac_phase", "value": 0.0, "unit": "deg", "source": "interactive default"},

        {"tab": "Calibration", "control": "w_auto_w0", "value": False, "unit": "-", "source": "explicit model value"},
        {"tab": "Calibration", "control": "w_omega0", "value": float(cpp_model_engineering.omega0 / (2.0 * np.pi * 1e9)), "unit": "GHz", "source": "model omega0"},
        {"tab": "Calibration", "control": "w_n", "value": float(cpp_model_engineering.N), "unit": "-", "source": "model N"},
        {"tab": "Calibration", "control": "w_domega0_dBz", "value": float(cpp_model_engineering.field_cal.domega0_dBz / (2.0 * np.pi * 1e9)), "unit": "GHz/T", "source": "field calibration"},
        {"tab": "Calibration", "control": "w_seq_per_T", "value": float(cpp_model_engineering.field_cal.seq_per_T), "unit": "1/T", "source": "field calibration"},

        {"tab": "Solver", "control": "w_sim_mode", "value": "Fast+Full", "unit": "-", "source": "notebook exposes both"},
        {"tab": "Solver", "control": "w_tend", "value": float(t_end * 1e9), "unit": "ns", "source": "trajectory length"},
        {"tab": "Solver", "control": "w_dt", "value": float(dt * 1e12), "unit": "ps", "source": "trajectory dt"},
        {"tab": "Solver", "control": "w_sde", "value": False, "unit": "-", "source": "not enabled here"},
        {"tab": "Solver", "control": "w_temp", "value": 300.0, "unit": "K", "source": "interactive default"},
        {"tab": "Solver", "control": "w_noise", "value": 1.0, "unit": "-", "source": "interactive default"},
        {"tab": "Solver", "control": "w_rand_seed", "value": False, "unit": "-", "source": "interactive default"},
        {"tab": "Solver", "control": "w_seed", "value": 42, "unit": "-", "source": "interactive default"},
    ]
    interactive_params_df = pd.DataFrame(interactive_rows)

    cpp_internal_df = pd.DataFrame(
        [
            {"parameter": "sigma", "value": float(cpp_model_engineering._sigma), "unit": "(model internal)", "note": "Slonczewski prefactor"},
            {"parameter": "d0", "value": float(cpp_model_engineering._d0), "unit": "-", "note": "Linear damping coefficient"},
            {"parameter": "d1", "value": float(cpp_model_engineering._d1), "unit": "-", "note": "Nonlinear damping coefficient"},
            {"parameter": "chi_prefactor", "value": float(cpp_model_engineering._chi_prefactor), "unit": "rad/s per (A/m^2)", "note": "chi(J)=chi_prefactor*J*chi_scale"},
            {"parameter": "chi_scale", "value": float(cpp_model_engineering.chi_scale), "unit": "-", "note": "Global STT scaling"},
            {"parameter": "threshold_current_dc", "value": float(cpp_model_engineering.threshold_current_dc()), "unit": "A/m^2", "note": "Model auto-oscillation threshold"},
        ]
    )

    print("\nInteractive Thiele parameter mirror (dashboard controls):")
    display(interactive_params_df)

    print("CPP internal coefficients used by the analytical solver:")
    display(cpp_internal_df)

except Exception as exc:
    print("Engineering current sweep failed:", repr(exc))


## 12) Summary and practical notes

Short answer to: "Can I get practical Thiele modeling outputs directly from the notebook (without opening the interactive dashboard)?"

- Yes: the notebook now auto-builds an engineering current sweep (`f(J)`, `u0(J)`, pump-vs-damping balance) and reports critical currents.
- Yes: it also solves the inverse problem (`target f -> optimal J`) in the same calibrated model.
- Yes: all interactive-dashboard control parameters are dumped in a table (with value + source), plus internal CPP coefficients (`d0`, `d1`, `sigma`, `chi_prefactor`).
- Fit/forecast quality still depends on input data quality and on whether the operating regime matches the CPP Thiele assumptions.


In [ ]:
fj_points_n = int(fj_points_fit.shape[0]) if isinstance(fj_points_fit, pd.DataFrame) else 0
fj_fit_ok = thiele_fj_fit is not None

summary = {
    "zarr_path": ZARR_PATH,
    "n_time_samples": int(traj.time.size),
    "tracking_method": traj.method,
    "rotation_sense": traj.rotation_sense,
    "gyration_peak_ghz": float(gyr.peak_frequency_ghz),
    "breathing_peak_ghz": float(breath.peak_frequency_ghz),
    "topology_Q_mid": float(topo_by_frame[frame_ids[len(frame_ids) // 2]].Q),
    "st_f0_ghz": float(st.f_0_ghz),
    "st_linewidth_mhz": float(st.linewidth_hz * 1e-6),
    "bridge_delta_f_hz": float(cmp_proxy.metrics.delta_f_mean),
    "thiele_fJ_source": fj_fit_source if fj_points_fit is not None else "insufficient_data",
    "thiele_fJ_points": fj_points_n,
    "thiele_fJ_rmse_mhz": float(thiele_fj_fit.rmse_hz * 1e-6) if fj_fit_ok else float("nan"),
    "thiele_fJ_omega0_fit_rad_s": float(thiele_fj_fit.omega0) if fj_fit_ok else float("nan"),
    "thiele_fJ_N_fit": float(thiele_fj_fit.N) if fj_fit_ok else float("nan"),
    "thiele_fJ_domega0_dJ_fit": float(thiele_fj_fit.domega0_dJ) if fj_fit_ok else float("nan"),
    "thiele_fJ_chi_scale_fit": float(thiele_fj_fit.chi_scale) if fj_fit_ok else float("nan"),
    "cpp_fitted_delta_f_hz": (
        float(cmp_cpp_fitted.metrics.delta_f_mean)
        if cmp_cpp_fitted is not None
        else float("nan")
    ),
    "thiele_engineering_model_source": thiele_engineering_model_source,
    "engineering_sweep_points": int(engineering_sweep_df.shape[0]) if isinstance(engineering_sweep_df, pd.DataFrame) else 0,
    "critical_J_threshold_MA_cm2": float(j_threshold_a_per_m2 * 1e-10) if np.isfinite(j_threshold_a_per_m2) else float("nan"),
    "critical_J_onset_scan_MA_cm2": float(j_onset_scan_a_per_m2 * 1e-10) if np.isfinite(j_onset_scan_a_per_m2) else float("nan"),
    "critical_J_edge_MA_cm2": float(j_edge_a_per_m2 * 1e-10) if np.isfinite(j_edge_a_per_m2) else float("nan"),
    "critical_operating_margin_MA_cm2": float(j_operating_margin_a_per_m2 * 1e-10) if np.isfinite(j_operating_margin_a_per_m2) else float("nan"),
    "inverse_design_points": int(target_currents_df.shape[0]) if isinstance(target_currents_df, pd.DataFrame) else 0,
    "interactive_param_rows": int(interactive_params_df.shape[0]) if isinstance(interactive_params_df, pd.DataFrame) else 0,
    "energy_channels": ", ".join(energy_ts.available_channels) if energy_ts.available_channels else "none",
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

print("\nAuto-inference coverage in this state:")
print("- material params (Ms, alpha, P, A): YES (from attrs, with defaults)")
print("- geometry (R, L): YES (from dataset shape + attrs)")
print("- polarity/chirality: YES/PARTIAL (polarity from attrs, chirality from attrs or default)")
print("- current waveform for analytical model: PARTIAL (table J aliases or fallback 0)")
print(
    "- CPP f(J) auto-calibration: "
    + ("YES (omega0, N, domega0_dJ, chi_scale fitted)" if fj_fit_ok else "NO (insufficient valid J-f points)")
)
print("- engineering sweep + critical currents: YES")
print("- interactive-dashboard parameter mirror table: YES")
print("- best-fit Thiele dynamics to numerics: PARTIAL (fit quality remains data- and model-regime dependent)")
